In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

In [2]:
# ─────────────────────────────────────────────
# 1. 配置
# ─────────────────────────────────────────────
resnet_mean = [0.485, 0.456, 0.406]
resnet_std  = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备：{device}")

使用设备：cuda


In [3]:
# ─────────────────────────────────────────────
# 2. 准备模型（必须在 optimized_extract 调用前完成）
# ─────────────────────────────────────────────
resnet_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet_model.fc = nn.Identity()
resnet_model.to(device)
resnet_model.eval()  # 固定 BN/Dropout，保证特征提取稳定性
print("准备ResNet18完毕")

准备ResNet18完毕


In [4]:
# ─────────────────────────────────────────────
# 3. 特征提取函数
# ─────────────────────────────────────────────


@torch.no_grad()
def fast_extract(imgs_np, batch_size=1024):
    """
    最快方案：
    - 一次性把全部图像转为 Tensor
    - 直接在 GPU 上分批推理，无 DataLoader 开销
    - FP16 autocast 加速
    - 预分配输出 tensor
    """
    resnet_model.eval()

    all_imgs = torch.from_numpy(imgs_np).permute(0, 3, 1, 2).float().div_(255.0)

    # 提前移到 GPU，循环内不再重复 .to(device)
    mean = torch.tensor(resnet_mean).view(1, 3, 1, 1).to(device)
    std  = torch.tensor(resnet_std).view(1, 3, 1, 1).to(device)

    N = all_imgs.shape[0]
    all_features = torch.zeros(N, 512, dtype=torch.float32)

    for i in range(0, N, batch_size):
        batch = all_imgs[i:i + batch_size].to(device)
        batch = F.interpolate(batch, size=(224, 224),
                              mode='bilinear', align_corners=False)
        batch.sub_(mean).div_(std)          # in-place 归一化，避免新建临时 tensor

        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            feat = resnet_model(batch)

        all_features[i:i + batch_size] = feat.float().cpu()

    return all_features

In [5]:
# ─────────────────────────────────────────────
# 4. 加载 CIFAR-10 原始数据并提取特征
# ─────────────────────────────────────────────

train_data = datasets.CIFAR10(root='C:/Jupyter(Anaconda)/data', train=True,  download=False)
test_data  = datasets.CIFAR10(root='C:/Jupyter(Anaconda)/data', train=False, download=False)

print("正在提取训练集特征...")
X_train = fast_extract(train_data.data)   # (50000, 512)
y_train = torch.tensor(train_data.targets)

print("正在提取测试集特征...")
X_test  = fast_extract(test_data.data)    # (10000, 512)
y_test  = torch.tensor(test_data.targets)

正在提取训练集特征...
正在提取测试集特征...


In [6]:
# ─────────────────────────────────────────────
# 5. 特征标准化（修复：避免 MLP 输入数值范围不均匀）
# ─────────────────────────────────────────────
feat_mean = X_train.mean(dim=0)
feat_std  = X_train.std(dim=0) + 1e-8          # 防止除零

X_train = (X_train - feat_mean) / feat_std
X_test  = (X_test  - feat_mean) / feat_std     # 用训练集统计量归一化测试集

In [7]:
# ─────────────────────────────────────────────
# 6. 定义更稳健的 MLP
# ─────────────────────────────────────────────
class MLPHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 512),
            nn.BatchNorm1d(512),    # 加入 BN 层提高稳定性
            nn.ReLU(),
            nn.Dropout(0.3),        # 防止过拟合
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.net(x)

model = MLPHead().to(device)

In [8]:
# ─────────────────────────────────────────────
# 7. 优化器、调度器、损失函数
# ─────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

# 余弦退火调度器：随 epoch 平滑降低学习率
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# ─────────────────────────────────────────────
# 8. DataLoader（修复：drop_last=True 防止 BN 在 batch=1 时崩溃）
# ─────────────────────────────────────────────
train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=250,
    shuffle=True,
    drop_last=True,                 # 丢弃最后不足 batch_size 的样本
    pin_memory=(device.type == "cuda")
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=500,
    shuffle=False,
    pin_memory=(device.type == "cuda")
)

In [9]:
# ─────────────────────────────────────────────
# 9. 训练循环（修复：原代码完全缺失）
# ─────────────────────────────────────────────
def evaluate(loader):
    """返回在给定 loader 上的准确率"""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            preds = model(X_batch).argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total   += y_batch.size(0)
    return correct / total

num_epochs = 10
##best_acc   = 0.0

for epoch in range(1, num_epochs + 1):
    model.train()#
    total_loss = 0.0
    correct    = 0
    total      = 0

    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = loss_fn(logits, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * y_batch.size(0)
        correct    += (logits.argmax(dim=1) == y_batch).sum().item()
        total      += y_batch.size(0)

    scheduler.step()

    train_loss = total_loss / total
    train_acc  = correct    / total
    #test_acc   = evaluate(test_loader)

    # 保存最优模型
    #if test_acc > best_acc:
        #best_acc = test_acc
        #torch.save(model.state_dict(), "best_mlp.pth")

    print(f"Epoch [{epoch:02d}/{num_epochs}]  "
          f"Loss: {train_loss:.4f}  "
          f"Train Acc: {train_acc:.4f}  "
          #f"Test Acc: {test_acc:.4f}  "
          f"lr: {scheduler.get_last_lr()[0]:.2e}")

print(f"\n训练完成！")

Epoch [01/10]  Loss: 0.9414  Train Acc: 0.7378  lr: 9.76e-05
Epoch [02/10]  Loss: 0.4930  Train Acc: 0.8491  lr: 9.05e-05
Epoch [03/10]  Loss: 0.4163  Train Acc: 0.8666  lr: 7.94e-05
Epoch [04/10]  Loss: 0.3807  Train Acc: 0.8752  lr: 6.55e-05
Epoch [05/10]  Loss: 0.3599  Train Acc: 0.8796  lr: 5.00e-05
Epoch [06/10]  Loss: 0.3452  Train Acc: 0.8842  lr: 3.45e-05
Epoch [07/10]  Loss: 0.3356  Train Acc: 0.8880  lr: 2.06e-05
Epoch [08/10]  Loss: 0.3287  Train Acc: 0.8900  lr: 9.55e-06
Epoch [09/10]  Loss: 0.3252  Train Acc: 0.8918  lr: 2.45e-06
Epoch [10/10]  Loss: 0.3223  Train Acc: 0.8921  lr: 0.00e+00

训练完成！


In [10]:
print("正在评估测试集...")
print(f"测试集准确率：{evaluate(test_loader)*100:.2f}%")

正在评估测试集...
测试集准确率：87.32%
